<a href="https://colab.research.google.com/github/Paupiere/Restaurant-Review-Sentiment-Analysis/blob/main/Sentiment_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. IMPORT NECESSARY LIBRARIES
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Download NLTK stop words (runs quietly in Colab)
nltk.download('stopwords', quiet=True)

# 2. CREATE THE EXPANDED MAJITAR RESTAURANT DATASET
df = pd.read_csv('dataset.csv')

# 3. TEXT PREPROCESSING FUNCTION
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower() # Lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove punctuation and numbers
    words = text.split()
    words = [word for word in words if word not in stop_words] # Remove stopwords
    return ' '.join(words)

df['Cleaned_Review'] = df['Review'].apply(clean_text)

# 4. FEATURE EXTRACTION & MODEL TRAINING
# Convert text to numbers using TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['Cleaned_Review'])
y = df['True_Sentiment']

# Split data into training and testing sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train the Logistic Regression Model
model = LogisticRegression()
model.fit(X_train, y_train)

# 5. INTERACTIVE FUNCTION FOR YOUR ASSIGNMENT PRESENTATION
def analyze_restaurant():
    # Automatically get unique restaurant names for the prompt
    unique_restaurants = ", ".join(df['Restaurant'].unique())
    print(f"\nAvailable Restaurants: {unique_restaurants}")
    restaurant_name = input("\nEnter a restaurant name from the list above: ")

    # Filter dataset for the chosen restaurant
    restaurant_data = df[df['Restaurant'].str.lower() == restaurant_name.lower()]

    if restaurant_data.empty:
        print("Restaurant not found in the dataset. Please check the spelling.")
        return

    print(f"\n--- Analyzing Reviews for: {restaurant_name.title()} ---\n")

    # Predict sentiments for the specific restaurant
    reviews_vectorized = vectorizer.transform(restaurant_data['Cleaned_Review'])
    predictions = model.predict(reviews_vectorized)

    # Display the results
    for i, row in enumerate(restaurant_data.itertuples()):
        sentiment_label = "Positive" if predictions[i] == 1 else "Negative"
        print(f"Review: \"{row.Review}\"")
        print(f"Predicted Sentiment: {sentiment_label}\n")

# 6. EVALUATE MODEL PERFORMANCE
def evaluate_model():
    y_pred = model.predict(X_test)

    print("=========================================")
    print("      MODEL PERFORMANCE EVALUATION       ")
    print("=========================================")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.2f}")
    print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.2f}")
    print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.2f}")
    print(f"F1-Score:  {f1_score(y_test, y_pred, zero_division=0):.2f}")
    print("=========================================\n")

# --- EXECUTE THE FUNCTIONS ---
evaluate_model()
analyze_restaurant()

      MODEL PERFORMANCE EVALUATION       
Accuracy:  0.67
Precision: 0.67
Recall:    1.00
F1-Score:  0.80


Available Restaurants: Punjabi Kitchen, Grill and Chill Majhitar, Coffee Break, Imperfecto Restro, Down Town Restro, Lounge 88, Paddington, Biryani Kitchen

Enter a restaurant name from the list above: Coffee Break

--- Analyzing Reviews for: Coffee Break ---

Review: "A gem for anyone looking for delicious treats at reasonable prices."
Predicted Sentiment: Positive

Review: "Pastries are soft and flavorful, worth every penny."
Predicted Sentiment: Positive

Review: "Stale food, total waste of money. Will not visit again."
Predicted Sentiment: Positive

Review: "The birthday cake I ordered was dry and not fresh at all."
Predicted Sentiment: Negative

Review: "Amazing cookies and patties. Very cozy ambiance."
Predicted Sentiment: Positive

